# NLP Preprocessing
This notebook cleans the raw Touch 'n Go Google Play reviews and exports `cleaned_data.csv`.

In [1]:
!pip install pandas nltk contractions

In [2]:
import pandas as pd
import re
import string
import os
import nltk
import contractions

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

Download NLTK resources

In [3]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

Set file paths

In [4]:
RAW_DATA_PATH = "../data/raw_data/touch_n_go_reviews_raw.csv"
CLEANED_DATA_PATH = "../data/cleaned_data.csv"
os.makedirs("../data", exist_ok=True)

Load raw dataset

In [5]:
df = pd.read_csv(RAW_DATA_PATH)
print(df.shape)
df.head()

(20000, 6)


,review_id,review_text,rating,thumbs_up_count,review_datetime,sentiment_label
0,7bdb015a-5670-4177-8379-b7a2f5f7a35b,new update really troublesome! remove the impo...,1,0,2026-06-29 21:45:10,negative
1,995f7b18-1ded-443e-bc8f-6592f7f8bd81,good,5,0,2026-06-29 21:23:25,positive
2,8b79bed1-74c9-42e2-8749-b682baecd60b,One of the most horrible UI design I've ever e...,1,0,2026-06-29 20:49:33,negative
3,5c58b099-e276-4769-937d-9bb3efcd00b8,Bloatware. UI front page now focused on sellin...,1,0,2026-06-29 19:03:11,negative
4,cb9db6de-88d9-4b19-8999-646665ee8e99,"After update, changing of the UI is still fine...",1,0,2026-06-29 17:22:46,negative


Basic checking

In [6]:
print(df.info())
print(df.isna().sum())
print(df["sentiment_label"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   review_id        20000 non-null  object
 1   review_text      20000 non-null  object
 2   rating           20000 non-null  int64 
 3   thumbs_up_count  20000 non-null  int64 
 4   review_datetime  20000 non-null  object
 5   sentiment_label  20000 non-null  object
dtypes: int64(2), object(4)
memory usage: 937.6+ KB
None
review_id          0
review_text        0
rating             0
thumbs_up_count    0
review_datetime    0
sentiment_label    0
dtype: int64
sentiment_label
positive    14014
negative     5226
neutral       760
Name: count, dtype: int64


Prepare English + Malay stopwords

In [7]:
english_stopwords = set(stopwords.words("english"))

# Custom Malay/English stopwords based on common Google Play review wording
custom_stopwords = {
    # Malay common words
    "yang", "dan", "ini", "itu", "untuk", "dengan", "dari", "pada", "dalam",
    "saya", "aku", "kami", "kita", "dia", "mereka", "anda", "kamu",
    "ada", "jadi", "akan", "sudah", "dah", "telah", "masih", "belum",
    "boleh", "nak", "mahu", "perlu", "kena", "guna", "pakai",
    "sebab", "kerana", "kalau", "jika", "tapi", "tetapi", "namun",
    "pun", "lah", "kan", "je", "saja", "sahaja", "ni", "tu",

    # Malay short forms
    "yg", "dgn", "utk", "dlm", "pd", "dr", "sbb", "krn", "jgn",
    "sy", "aq", "korang", "diorang",

    # App review filler words
    "app", "apps", "application", "tng", "touch", "go", "ewallet",
    "please", "pls", "plz", "thank", "thanks", "ok", "okay"
}

stopword_list = english_stopwords.union(custom_stopwords)

lemmatizer = WordNetLemmatizer()

Malay slang normalization dictionary

In [8]:
# Custom normalization dictionary for Malaysian-style app reviews
TEXT_NORMALIZATION = {
    # Malay negation and short forms
    r"\btak\b": "tidak",
    r"\btk\b": "tidak",
    r"\btx\b": "tidak",
    r"\bx\b": "tidak",
    r"\bxde\b": "tiada",
    r"\btakde\b": "tidak ada",
    r"\btkde\b": "tidak ada",
    r"\btade\b": "tiada",

    # Common Malay abbreviations
    r"\bdpt\b": "dapat",
    r"\bblh\b": "boleh",
    r"\bboleh2\b": "boleh",
    r"\bsgt\b": "sangat",
    r"\bbg\b": "bagi",
    r"\bbrg\b": "barang",
    r"\bmsuk\b": "masuk",

    # English app review shortcuts
    r"\bpls\b": "please",
    r"\bplz\b": "please",
    r"\bmsg\b": "message",
    r"\botp\b": "otp",
    r"\blogin\b": "log in",
    r"\bloggedin\b": "logged in",

    # Common noisy expressions
    r"\bhaha+\b": "",
    r"\bhehe+\b": "",
    r"\blol+\b": "",
    r"\bwah+\b": "",
    r"\bwei+\b": "",
    r"\bweh+\b": ""
}

Define preprocessing functions

In [9]:
def normalize_text_terms(text):
    for pattern, replacement in TEXT_NORMALIZATION.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text

def clean_text(text):
    if pd.isna(text):
        return ""
    text=str(text).lower()
    text=contractions.fix(text)
    text = normalize_text_terms(text)
    text=re.sub(r"http\S+|www\S+|https\S+"," ",text)
    text=re.sub(r"\S+@\S+"," ",text)
    text=re.sub(r"<.*?>"," ",text)
    text=text.encode("ascii","ignore").decode()
    text=re.sub(r"(.)\1{2,}",r"\1\1",text)
    text=re.sub(r"\d+"," ",text)
    text=text.translate(str.maketrans("","",string.punctuation))
    text=re.sub(r"\s+"," ",text).strip()
    tokens=text.split()
    tokens=[w for w in tokens if w not in stopword_list and len(w) > 2]
    tokens=[lemmatizer.lemmatize(w) for w in tokens]
    return " ".join(tokens)

Apply NLP preprocessing

In [10]:
df["cleaned_review"]=df["review_text"].apply(clean_text)
df[["review_text","cleaned_review"]].head(10)

,review_text,cleaned_review
0,new update really troublesome! remove the impo...,new update really troublesome remove important...
1,good,good
2,One of the most horrible UI design I've ever e...,one horrible design ever encountered designer ...
3,Bloatware. UI front page now focused on sellin...,bloatware front page focused selling product b...
4,"After update, changing of the UI is still fine...",update changing still fine via duit slow need ...
5,"The name of the app is Touch 'n Go. However, t...",name however function top burried tonne featur...
6,updated version is very usable and have a lot ...,updated version usable lot function transactio...
7,Stop bombarding people with promotional ads or...,stop bombarding people promotional ad least ma...
8,design is not user friendly. SACK THE BOSSES!!...,design user friendly sack boss stupid old people
9,Bad UX,bad


Remove empty cleaned reviews, add review length columns

In [11]:
df=df[df["cleaned_review"].str.strip()!=""]
df["raw_review_length"]=df["review_text"].astype(str).str.len()
df["cleaned_review_length"]=df["cleaned_review"].astype(str).str.len()
df["cleaned_word_count"]=df["cleaned_review"].str.split().str.len()
print(df.shape)
df.head()

(18835, 10)


,review_id,review_text,rating,thumbs_up_count,review_datetime,sentiment_label,cleaned_review,raw_review_length,cleaned_review_length,cleaned_word_count
0,7bdb015a-5670-4177-8379-b7a2f5f7a35b,new update really troublesome! remove the impo...,1,0,2026-06-29 21:45:10,negative,new update really troublesome remove important...,119,87,12
1,995f7b18-1ded-443e-bc8f-6592f7f8bd81,good,5,0,2026-06-29 21:23:25,positive,good,4,4,1
2,8b79bed1-74c9-42e2-8749-b682baecd60b,One of the most horrible UI design I've ever e...,1,0,2026-06-29 20:49:33,negative,one horrible design ever encountered designer ...,97,52,7
3,5c58b099-e276-4769-937d-9bb3efcd00b8,Bloatware. UI front page now focused on sellin...,1,0,2026-06-29 19:03:11,negative,bloatware front page focused selling product b...,212,167,24
4,cb9db6de-88d9-4b19-8999-646665ee8e99,"After update, changing of the UI is still fine...",1,0,2026-06-29 17:22:46,negative,update changing still fine via duit slow need ...,235,141,22


Save cleaned dataset

In [12]:
df.to_csv(CLEANED_DATA_PATH,index=False,encoding="utf-8-sig")
print("Saved:",CLEANED_DATA_PATH)

Saved: ../data/cleaned_data.csv
